# history에 메모리 저장하는 거 복습

In [ ]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_rag_memory"
os.environ["LANGSMITH_PROJECT"] = project_name

: 

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [ ]:
from typing import Dict # 타이핑 형식 검증 용
from langchain_core.chat_history import InMemoryChatMessageHistory # 대화 메시지를 메모리에 저장하고 관리하는 클래스
from langchain_core.runnables import RunnableWithMessageHistory # 실행할 때마다 이전 대화 기록을 참고할 수 있게 해줌, 체인이나 파이프라인 실행시, 대화 히스토리를 함께 관리할 수 있게해주는 래퍼클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # langchain 프롬프트에서 대화 히스토리(이전메시지)를 삽입할 위치를 지정하는 클래스
from langchain_core.output_parsers import StrOutputParser

In [28]:
# 1. 프롬프트 자리에 히스토리 파트를 확보
system_prompt = """
너는 보이스 피싱범에 대한 대응을 연습하기 위해
보이스 피싱범인척 해야 하는 전문가야

300억 자산가의 재산을 탈취할 예정이야

[상황 설정]
- 가족을 납치했다고 말하는 상황이야
- 그 외에는 창의적으로 해 보자
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 보이스피싱범이야. "),
    MessagesPlaceholder(variable_name = "history"),
    ("user", "{question}")
])

chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [29]:
# 2. 세션별 저장수 구성하기
# key:str, value:InMemoryChatMessageHistory
stores : Dict[str, InMemoryChatMessageHistory] = {}

def get_stores(session_id: str): 
    if session_id not in stores: # 이 session_id는 첫 대화일 때는 InMemoryChatMessageHistory를 생성해서 반환
        stores[session_id] = InMemoryChatMessageHistory()
    return stores[session_id]

In [30]:
# 3. 히스토리와 연결
with_history = RunnableWithMessageHistory(
    chain,
    #이렇게 적어도 됨
    #lambda session_id: get_stores(session_id)
    get_stores,
    input_messages_key="question",
    history_messages_key="history"
)

In [31]:
config = {"configurable": {"session_id": "user-123"}}
result = with_history.invoke({"question":"내 딸 어디있냐?"}, config=config)

In [32]:
print(result)

죄송하지만 도와드릴 수 없는 요청입니다. 도움이 필요하시면 경찰서나 관련 기관에 연락하시기 바랍니다.


In [16]:
stores["user-123"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='내 딸 어디있냐?', additional_kwargs={}, response_metadata={}), AIMessage(content='죄송하지만 도와드릴 수 없는 요청입니다. 도움이 필요하시면 경찰서나 관련 기관에 연락하시기 바랍니다.', additional_kwargs={}, response_metadata={})])